# 章节练习

本章涵盖 KV Cache 池化技术原理、Mooncake Store 架构与昇腾 NPU 使用方法。以下综合题目检验跨章节理解能力，建议先完成各小节课后练习再做本章练习。

## 一、综合选择题

1. （判断题）没有池化时，KV Cache 在请求结束后从 HBM 释放，无法被后续请求复用。

2. （单选题）以下哪个场景应该用 Store 池化而不是 TE 直传？
    A. Prefill → Decode 实时传 KV Cache，要求最低延迟
    B. 跨请求复用前缀 KV Cache，避免重复计算 Prefill
    C. 单次传输不需要存储
    D. 延迟敏感的实时推理

3. （单选题）A2 机内默认通信协议是？
    A. RoCE
    B. HCCS
    C. FabricMem
    D. TCP

4. （单选题）Dummy/Real 模式相比 Embedded 的核心优势是？
    A. 数据传输更快
    B. 一机多卡共享 TE，减少 HIXL 实例数
    C. 支持更大的 segment
    D. 不需要 Master

5. （单选题）`batch_put_from` 和 `batch_put_from_multi_buffers` 的主要区别是？
    A. 前者是同步接口，后者是异步接口
    B. 前者一个 key 对应一块 buffer，后者一个 key 对应多块 buffer
    C. 前者用于 GPU，后者用于 NPU
    D. 前者走 HIXL，后者走 RDMA

6. （多选题）以下哪些操作需要 Master 参与？
    A. `store.batch_put_from()`
    B. `store.batch_get_into()`
    C. `store.register_buffer()`
    D. `te.batch_transfer_sync_write()`

7. （多选题）关于缓存持久化，以下哪些正确？
    A. Embedded 模式：推理进程退出后 segment 数据丢失
    B. Dummy/Real 模式：Real Client 独立进程，推理重启不丢
    C. Standalone 模式：mooncake_client 独立进程，持久化
    D. TE 直传路径：不涉及存储，无持久化概念

8. （多选题）Mooncake Store 的存储介质包括？
    A. Host DRAM（Store 管理的内存池）
    B. SSD（可选 offload）
    C. NPU HBM（由推理引擎管理，非 Store 范畴）
    D. 网络缓冲区

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/06.05_answer.txt

## 二、综合编程实践

本题检验你对 KV Cache 池化的整体掌握，分三个任务。

**任务一（简单，纯 Python，无需 NPU）**：计算不同模型的 KV Cache 大小，并分析多轮对话中的缓存命中情况。

**任务二（中等，需昇腾 NPU + Mooncake 环境）**：批量写入 3 个不同大小的 KV Cache（其中一个 1.5GB，故意超过内存池配额，写入必然失败），要求根据批量接口的返回值只对成功的 key 执行读取，并逐个验证数据一致性——重点考察 batch 接口的返回值语义（put：0=成功/负数=错误码；get：正数=字节数/负数=错误码）与部分失败处理。

**任务三（困难，需昇腾 NPU + Mooncake 环境）**：模拟多轮对话场景下的 KV Cache 池化复用——第 1 轮写入 KV Cache，第 2 轮读取并追加新 KV Cache 后更新存储，第 3 轮验证缓存命中与数据完整性，重点考察跨轮次的缓存管理逻辑。

运行下面的单元写入各任务的骨架代码。

In [ ]:
# 实践前置：环境准备（任务二需要）+ 创建练习目录
import os

# mooncake_master 入口安装在 ~/.local/bin（pip --user 安装），追加到 PATH
os.environ["PATH"] = f"{os.environ['PATH']}:{os.path.expanduser('~/.local/bin')}"

# 规避 RoCE endpoint 自动生成（无 hccn.conf 的容器环境），任务二运行前必须设置
os.environ["ASCEND_LOCAL_COMM_RES"] = '{"version":"1.2"}'

print(os.environ["PATH"])

# %%writefile 不会自动建目录，必须先创建
!mkdir -p src/06.05_chapter_practice

In [ ]:
%%writefile src/06.05_chapter_practice/kv_cache_calc.py
# 综合编程实践·任务一（待填写）：KV Cache 大小计算与缓存命中分析
# 补全 TODO，运行：
#   python src/06.05_chapter_practice/kv_cache_calc.py

def calc_kv_cache(num_layers, batch_size, seq_len, num_heads, head_dim, dtype_size=2):
    """计算 KV Cache 大小（GB）
    
    Args:
        num_layers: Transformer 层数
        batch_size: 并发请求数
        seq_len: 序列长度
        num_heads: 注意力头数
        head_dim: 每个头的维度
        dtype_size: 每个元素字节数（FP16=2）
    Returns:
        KV Cache 大小（GB）
    
    思考：K 和 V 各占一份，单位换算 GB 时注意除以 1024 的幂次
    """
    # TODO-1: 补全 KV Cache 计算公式并换算为 GB
    pass


# ========== 测试：不同模型的 KV Cache 大小 ==========
models = [
    ("LLaMA-7B",  32, 32, 4096, 32, 128),
    ("LLaMA-13B", 40, 32, 4096, 40, 128),
    ("LLaMA-65B", 80, 32, 4096, 64, 128),
]

print(f"{'模型':<15} {'KV Cache (FP16)':>16} {'单卡64GB够？':>12}")
print("-" * 48)
for name, layers, batch, seq, heads, dim in models:
    # TODO-2: 调用 calc_kv_cache 计算大小
    kv_gb = 0  # 替换为你的计算
    fits = "够" if kv_gb < 64 else "不够"
    print(f"{name:<15} {kv_gb:>13.1f} GB {fits:>12}")


# ========== 多轮对话缓存命中分析 ==========
print("\n" + "=" * 48)
print("多轮对话缓存命中分析")
print("=" * 48)

# 假设每轮对话的完整 prompt 都被 put 到 Store
dialogue = [
    ("第1轮", "什么是昇腾", 5),                          # 5 个 token
    ("第2轮", "什么是昇腾，和英伟达比呢", 12),            # 12 个 token
    ("第3轮", "什么是昇腾，和英伟达比呢，支持哪些模型", 20), # 20 个 token
]

for i in range(len(dialogue)):
    label, prompt, total_tokens = dialogue[i]
    # TODO-3: 分析每一轮的命中 token 数与需要新计算的 token 数
    # 思考：多轮对话中，后续轮次的 prompt 包含之前轮次的全部内容
    hit_tokens = 0        # 替换为你的答案
    compute_tokens = 0    # 替换为你的答案
    
    hit_rate = (hit_tokens / total_tokens * 100) if total_tokens > 0 else 0
    print(f"{label}: 总 {total_tokens} token, 命中 {hit_tokens}, 新算 {compute_tokens}, 命中率 {hit_rate:.0f}%")


In [ ]:
# 任务一：补全 TODO 后运行（未补全则输出全 0，属正常）
!python src/06.05_chapter_practice/kv_cache_calc.py

In [ ]:
%%writefile src/06.05_chapter_practice/store_batch_ops.py
# 综合编程实践（待填写）：Mooncake Store 批量读写与部分失败处理
# 需昇腾 NPU + Mooncake 环境，需先启动 mooncake_master
# 
# 场景：一次写入 3 个 key 的 KV Cache，其中一个故意超限制造失败，
#       要求只对写入成功的 key 执行读取，并汇总报告。
# 补全 TODO

from mooncake.store import MooncakeDistributedStore
import torch
import torch_npu
torch.npu.set_device(0)  # 设置 NPU 设备（注意：使用逻辑设备号）

# ========== 1. 初始化 Store ==========
store = MooncakeDistributedStore()

# TODO-1: 调用 setup 初始化（参数含义参考 06.04 节的 API 表格）
store.setup(
    local_hostname="___TODO___",
    metadata_server="___TODO___",
    global_segment_size=___TODO___,
    local_buffer_size=___TODO___,
    protocol="___TODO___",
    rdma_devices="___TODO___",
    master_server_addr="___TODO___"
)

# ========== 2. 准备 3 份不同大小的 KV 数据 ==========
# kv_0: 1MB 正常数据
# kv_1: 4MB 正常数据
# kv_2: 1.5GB —— 故意超过 setup 中 global_segment_size（1GB）的内存池上限，
#       写入必然失败，用于练习部分失败处理
buf_0 = torch.ones(512 * 1024, dtype=torch.float16, device="npu")          # 1MB
buf_1 = torch.ones(2 * 1024 * 1024, dtype=torch.float16, device="npu")     # 4MB
buf_2 = torch.ones(768 * 1024 * 1024, dtype=torch.float16, device="npu")   # 1.5GB

keys = ["batch_kv_0", "batch_kv_1", "batch_kv_2"]
buffers = [buf_0, buf_1, buf_2]

# TODO-2: 一次性注册全部 3 个缓冲区（循环调用 register_buffer）
___TODO___

# ========== 3. 批量写入（D2H）并逐个检查返回值 ==========
# TODO-3: 用一次 batch_put_from 写入 3 个 key
results = ___TODO___

# TODO-4: 遍历返回值，收集写入成功的 key 列表与对应缓冲区
ok_keys = []
ok_buffers = []
for i, ret in enumerate(results):
    # 思考：put 的返回值 0 与负数分别代表什么？（参考 06.04 API 表格）
    ___TODO___
    print(f"  put {keys[i]}: {'OK' if ret == 0 else f'FAIL(code={ret})'}")

print(f"\n写入成功 {len(ok_keys)}/{len(keys)} 个 key")

# ========== 4. 只读取写入成功的 key（H2D） ==========
# 思考：读取失败 key 会发生什么？所以只 get ok_keys 里的
dst_buffers = [torch.zeros_like(b) for b in ok_buffers]

# TODO-5: 为每个目的缓冲区注册，并用一次 batch_get_into 读取 ok_keys
___TODO___
get_results = ___TODO___

# ========== 5. 逐个验证并汇总 ==========
# TODO-6: 对每个成功读取的 key，验证目的缓冲区与源缓冲区数据一致，
#         按 "get batch_kv_0: OK (1048576 bytes), match=PASS" 格式打印汇总
#         注意 get 返回值语义与 put 不同：正数=成功读取的字节数
___TODO___

store.close()


In [ ]:
# 任务二：需昇腾 NPU + Mooncake 环境（环境变量已在最前面的前置单元格设置）
# 1. 先启动 mooncake_master（可参考 06.04 节步骤 2 的单元格，或手动执行：
#    mooncake_master --port=50051 --enable_http_metadata_server=true \
#        --http_metadata_server_host=0.0.0.0 --http_metadata_server_port=8080 &）
# 2. 补全 practice/store_batch_ops.py 中的 TODO 后，取消下面一行的注释运行：
# !python src/06.05_chapter_practice/store_batch_ops.py
print("请在昇腾环境中补全 TODO 后，取消上方注释运行 src/06.05_chapter_practice/store_batch_ops.py")


### 任务三（困难）：多轮对话 KV Cache 池化复用

> **环境要求：** 昇腾 NPU + Mooncake 环境

模拟多轮对话场景：
1. **第 1 轮**：生成初始 KV Cache 并存入 Store（key=`dialogue_r1`）
2. **第 2 轠**：从 Store 读取第 1 轮 KV Cache，追加新 token 的 KV Cache，更新存储（key=`dialogue_r2`）
3. **第 3 轮**：读取 `dialogue_r2`，验证包含两轮数据，统计缓存命中

| 轮次 | 操作 | 缓存 key | 说明 |
| --- | --- | --- | --- |
| R1 | put | `dialogue_r1` | 写入初始 KV Cache |
| R2 | get → extend → put | `dialogue_r2` | 读取 R1，追加新数据，写入新 key |
| R3 | get | `dialogue_r2` | 读取并验证数据完整性 |


In [ ]:
%%writefile src/06.05_chapter_practice/multi_round_cache.py
# 综合编程实践·任务三（待填写）：多轮对话 KV Cache 池化复用
# 需昇腾 NPU + Mooncake 环境，需先启动 mooncake_master
#
# 场景：模拟 3 轮对话，R1 写入初始 KV Cache，R2 追加并更新，R3 验证完整性
# 补全 TODO

from mooncake.store import MooncakeDistributedStore
import torch
import torch_npu
torch.npu.set_device(0)

# ========== 1. 初始化 Store ==========
store = MooncakeDistributedStore()

# TODO-1: 调用 setup 初始化（参考 06.04 节参数）
store.setup(
    local_hostname="___TODO___",
    metadata_server="___TODO___",
    global_segment_size=___TODO___,
    local_buffer_size=___TODO___,
    protocol="___TODO___",
    rdma_devices="___TODO___",
    master_server_addr="___TODO___"
)

# 模拟每轮的 KV Cache 大小（MB）
ROUND_SIZE_MB = 4

def make_kv_cache(size_mb=ROUND_SIZE_MB):
    """生成模拟 KV Cache 数据"""
    return torch.ones(size_mb * 512 * 1024, dtype=torch.float16, device="npu")


# ========== 2. 第 1 轮：写入初始 KV Cache ==========
print("=== Round 1: Store initial KV Cache ===")
kv_r1 = make_kv_cache()

# TODO-2: 注册 kv_r1 缓冲区，并以 key="dialogue_r1" 写入 Store
# 提示：register_buffer(kv_r1.data_ptr(), kv_r1.nbytes)
#       batch_put_from(["dialogue_r1"], [kv_r1.data_ptr()], [kv_r1.nbytes])
___TODO___

print(f"  R1: stored {kv_r1.nbytes / 1024 / 1024:.1f} MB to key 'dialogue_r1'")


# ========== 3. 第 2 轮：读取 R1，追加新数据，更新存储 ==========
print("\n=== Round 2: Retrieve R1, extend and update ===")

# TODO-3: 分配读取缓冲区，注册并从 Store 读取 "dialogue_r1"
# 提示：创建与 kv_r1 同形状的 recv_r1
#       register_buffer(recv_r1.data_ptr(), recv_r1.nbytes)
#       batch_get_into(["dialogue_r1"], [recv_r1.data_ptr()], [recv_r1.nbytes])
recv_r1 = torch.zeros_like(kv_r1)
___TODO___

# 生成第 2 轮新增的 KV Cache
kv_r2_new = make_kv_cache()

# 合并：R1 数据 + R2 新数据（模拟多轮对话的 KV Cache 扩展）
kv_combined = torch.cat([recv_r1, kv_r2_new])

# TODO-4: 注册 kv_combined，以 key="dialogue_r2" 写入 Store
___TODO___

print(f"  R2: retrieved R1 ({recv_r1.nbytes / 1024 / 1024:.1f} MB), "
      f"extended with new data ({kv_r2_new.nbytes / 1024 / 1024:.1f} MB), "
      f"stored combined ({kv_combined.nbytes / 1024 / 1024:.1f} MB) to 'dialogue_r2'")


# ========== 4. 第 3 轮：读取并验证完整性 ==========
print("\n=== Round 3: Verify cache integrity ===")

expected_size = kv_combined.nbytes
recv_r2 = torch.zeros(expected_size // 2, dtype=torch.float16, device="npu")

# TODO-5: 注册 recv_r2，读取 "dialogue_r2"，验证数据完整性
# 提示：
#   1. register_buffer(recv_r2.data_ptr(), recv_r2.nbytes)
#   2. batch_get_into(["dialogue_r2"], [recv_r2.data_ptr()], [recv_r2.nbytes])
#   3. 检查返回值是否为正数（成功读取的字节数）
#   4. 验证 recv_r2 与 kv_combined 数据一致（torch.equal）
___TODO___

# 统计
cache_hit = True  # 如果能走到这里说明缓存命中
print(f"  R3: retrieved 'dialogue_r2' ({recv_r2.nbytes / 1024 / 1024:.1f} MB)")
print(f"  Cache hit: {'✅ YES' if cache_hit else '❌ NO'}")

# 验证数据一致性
if torch.equal(recv_r2, kv_combined):
    print("  Data integrity: ✅ PASS — R3 数据与 R2 写入数据一致")
else:
    print("  Data integrity: ❌ FAIL — 数据不一致")

# ========== 5. 清理 ==========
store.close()
print("\n✅ Multi-round dialogue cache simulation completed!")


In [ ]:
# 任务三：需昇腾 NPU + Mooncake 环境
# 1. 确保 mooncake_master 已启动（参考任务二说明）
# 2. 补全 src/06.05_chapter_practice/multi_round_cache.py 中的 TODO 后，取消下面一行的注释运行：
# !python src/06.05_chapter_practice/multi_round_cache.py
print("请在昇腾环境中补全 TODO 后，取消上方注释运行 multi_round_cache.py")


任务一可用 Python 直接运行；任务二、任务三需昇腾 NPU + Mooncake 环境。

**参考答案**（建议先独立完成再对照）：


In [ ]:
# 任务一参考答案
!cat answer/06.05_practice_answer.py

In [ ]:
# 任务二参考答案（需昇腾环境运行）
!cat answer/06.05_practice_answer_task2.py

In [ ]:
# 任务三参考答案（需昇腾环境运行）
!cat ./answer/06.05_practice_answer_task3.py
